# 3. Privacy & Governance Demo

This notebook addresses **privacy** and **governance** for the NovaCred credit dataset, aligned with Session 5 (Bias & Privacy in Data) and the project rubric: identify PII, demonstrate **pseudonymization** (or anonymization), and map findings to **GDPR** (lawful basis, data minimization, storage limitation, right to erasure). Uses the cleaned dataset from `01-data-quality.ipynb`.

In [ ]:
from pathlib import Path
import hashlib
import secrets
import pandas as pd

DATA_PATH = Path("../data/credit_applications_clean_final.csv")

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head(2)

## 3.1 Identify PII and quasi-identifiers

**Direct identifiers** (obviously identifying): name, email, SSN, IP address, phone.

**Quasi-identifiers** (Session 5): combinations can re-identify individuals. E.g. ZIP + gender + birth date (Sweeney 2000: 87% of Americans uniquely identifiable with just these three). In our schema:
- **Direct:** `applicant_info.full_name`, `applicant_info.email`, `applicant_info.ssn`, `applicant_info.ip_address`
- **Quasi-identifiers:** `applicant_info.zip_code`, `applicant_info.date_of_birth`, `applicant_info.gender`

Pseudonymizing only direct identifiers is not enough—if Lisbon has one person with a given salary and diagnosis, they can still be identified. For full anonymization we would need generalization/suppression of quasi-identifiers (e.g. k-anonymity).

In [ ]:
# PII columns in our dataset
direct_id = ["applicant_info.full_name", "applicant_info.email", "applicant_info.ssn", "applicant_info.ip_address"]
quasi_id = ["applicant_info.zip_code", "applicant_info.date_of_birth", "applicant_info.gender"]
print("Direct identifiers:", direct_id)
print("Quasi-identifiers:", quasi_id)
existing_direct = [c for c in direct_id if c in df.columns]
print("\nPresent in dataframe:", existing_direct)

## 3.2 Pseudonymization vs anonymization (Session 5)

| | Pseudonymization | Anonymization |
|--|------------------|---------------|
| **How it works** | Replace identifiers with tokens/hashes | Generalize or suppress (e.g. k-anonymity) |
| **Reversible?** | Yes (with key) | No (if done right) |
| **GDPR** | Still personal data | Not personal data |

**GDPR Article 4(5):** Pseudonymization means processing personal data so that it can no longer be attributed to a specific data subject **without additional information**, provided that such additional information is kept separately and is subject to technical and organizational measures.

**Warning:** Pseudonymization alone is NOT anonymization; data can still be re-identified with the mapping key.

## 3.3 Demonstrate pseudonymization: hashing with salt

Session 5: **Hashing** (e.g. SHA-256) is common. Same input → same hash (deterministic), so an attacker who knows possible values can brute-force. **Fix: add a salt** (random string, stored separately) before hashing so pre-computation is infeasible.

We pseudonymize one PII column (e.g. email) using SHA-256 with a salt. In production the salt would be stored securely and separately.

In [ ]:
def pseudonymize_sha256(value: str, salt: str) -> str:
    """Hash value with salt so same input does not yield same output across datasets (salt secret)."""
    if pd.isna(value):
        return ""
    return hashlib.sha256((salt + str(value).strip()).encode("utf-8")).hexdigest()

# Salt: in production, store securely and separately (e.g. secrets management)
SALT = secrets.token_hex(16)
col = "applicant_info.email"
df_pseudo = df.copy()
df_pseudo["email_pseudonymized"] = df_pseudo[col].apply(lambda x: pseudonymize_sha256(x, SALT))
print("Example: first non-null email -> pseudonymized")
sample = df_pseudo[df_pseudo[col].notna()].iloc[0]
print(f"  Original: {sample[col]}")
print(f"  Pseudonymized: {sample['email_pseudonymized'][:24]}...")

In [ ]:
# Show before/after for a few rows (hide original in report if needed)
display_cols = [col, "email_pseudonymized"]
df_pseudo[display_cols].head(3)

## 3.4 Map to GDPR and governance

- **Lawful basis (Art. 6):** Credit assessment typically relies on contract performance or legitimate interest; consent may apply for marketing. NovaCred should document the basis for each processing purpose.
- **Data minimization (Art. 5(1)(c)):** Collect only what is necessary. PII such as full name, email, SSN, IP should be justified; consider whether all fields are needed for the decision.
- **Storage limitation (Art. 5(1)(e)):** Retain application data only as long as necessary for the purpose (e.g. legal obligation, dispute, audit). Define retention periods and automate deletion where possible.
- **Right to erasure (Art. 17):** Pseudonymized data is still personal data; erasure requests apply. The mapping (e.g. email → hash) must allow deletion of the data subject’s records when required.

**EU AI Act:** Credit scoring is high-risk AI. Requirements include transparency, human oversight, risk management, and data governance—aligning with bias monitoring (Notebook 02) and privacy controls here.